<a href="https://colab.research.google.com/github/UKD1211/Tier_N_Supplier_Visibility_Risk_Pred/blob/main/supplier_risk_pred_main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np


In [ ]:
df_main= pd.read_csv("/content/final_dataset_main.csv")

In [ ]:
# df_main.drop(columns = ['Unnamed: 0'], inplace = True)

In [ ]:
df_main

In [ ]:
df_main['high_risk'].value_counts(normalize = True)*100

In [ ]:
df_main.drop(columns = ['supplier_id', 'state', 'cluster', 'risk_score', 'stage'], inplace = True)

In [ ]:
df_main

In [ ]:
X = df_main.drop(columns = ['high_risk'])
y = df_main['high_risk']

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.35, random_state = 42, stratify = y)

In [ ]:
X_train

In [ ]:
y_train.value_counts(normalize = True)*100

In [ ]:
y_test.value_counts(normalize = True)*100

In [ ]:
#EDA #distribution of the risk here

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
sns.set_theme(style="whitegrid", context="talk")

In [ ]:
test_df = pd.read_csv("/content/final_dataset_main.csv")

In [ ]:
sns.kdeplot(test_df['risk_score'])

In [ ]:
df_main['high_risk'].value_counts().plot(kind = 'bar')

In [ ]:
plt.figure(figsize=(8,5))

sns.barplot(
    x='tier',
    y='risk_score',
    data=test_df,
    palette='viridis'   # smooth professional gradient
)

plt.title("Average Risk Score Across Supply Chain Tiers", fontsize=14, weight='bold')
plt.xlabel("Tier Level")
plt.ylabel("Average Risk Score")

plt.tight_layout()
plt.savefig("tier_vs_risk_bar.png", dpi=300)  # save for PPT
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(
    x='tier',
    y='risk_score',
    data=test_df,
    palette='Set2'   # distinct colors for tiers
)

plt.title("Risk Distribution Across Supply Chain Tiers", fontsize=14, weight='bold')
plt.xlabel("Tier Level")
plt.ylabel("Risk Score")

plt.tight_layout()
plt.savefig("tier_vs_risk_box.png", dpi=300)
plt.show()

In [ ]:
pd.crosstab(test_df['tier'], test_df['high_risk'])

In [ ]:
plt.figure(figsize=(10,6))
df_corr = test_df.drop(columns=['supplier_id', 'state', 'cluster', 'stage', 'high_risk'])
sns.heatmap(df_corr.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Feature Correlation Matrix", weight='bold')
plt.tight_layout()
plt.savefig("correlation_heatmap.png", dpi=300)
plt.show()

In [ ]:
plt.figure(figsize=(10,6))

df_corr = test_df.drop(columns=[
    'supplier_id', 'state', 'cluster', 'stage', 'high_risk'
])

corr = df_corr.corr()

# Create mask for upper triangle (including diagonal)
mask = np.triu(np.ones_like(corr, dtype=bool))

sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    cmap='coolwarm',
    fmt=".2f",
    linewidths=0.5,
    cbar_kws={"shrink": 0.8}
)

plt.title("Feature Correlation Matrix", weight='bold')
plt.tight_layout()
plt.savefig("correlation_lower_triangle.png", dpi=300)
plt.show()

In [ ]:
print(sns.scatterplot(data = test_df, y = 'risk_score', x = 'tier'))

In [ ]:
sns.scatterplot(x='visibility', y='risk_score', data=test_df)
plt.title("Visibility vs Risk Score", weight='bold')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

features = [
    'labour_intensity',
    'informality',
    'visibility',
    'cost_pressure',
    'state_risk_cal',
    'risk_score'
]

plt.figure(figsize=(15,10))

for i, feature in enumerate(features, 1):
    plt.subplot(2, 3, i)

    sns.boxplot(
        x='tier',
        y=feature,
        data=test_df,
        palette='Set2'
    )

    plt.title(f"{feature.replace('_',' ').title()} vs Tier", fontsize=11, weight='bold')
    plt.xlabel("Tier")
    plt.ylabel(feature)

plt.tight_layout()
plt.savefig("tier_vs_features_boxplots.png", dpi=300)
plt.show()

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns

# features only - use 'test_df' which is the original dataframe
X_pca_features = test_df.drop(columns=['supplier_id', 'risk_score', 'high_risk',
                                     'state', 'cluster', 'stage'])

# scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_pca_features)

# PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# create dataframe
pca_df = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
# Get high_risk from the original test_df
pca_df['high_risk'] = test_df['high_risk']

# plot
plt.figure(figsize=(8,6))
sns.scatterplot(data=pca_df, x='PC1', y='PC2', hue='high_risk', palette='Set1')
plt.title("PCA Projection of Suppliers", weight='bold')
plt.show()

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,confusion_matrix,classification_report,precision_recall_curve,roc_auc_score
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV

In [ ]:
lor = LogisticRegression(max_iter=1000)
rfc = RandomForestClassifier(n_estimators=100)
dtc = DecisionTreeClassifier()
gbc = GradientBoostingClassifier()

In [ ]:
lor.fit(X_train, y_train)
rfc.fit(X_train, y_train)
dtc.fit(X_train, y_train)
gbc.fit(X_train, y_train)

In [ ]:
y_pred_lor = lor.predict(X_test)
y_pred_rfc = rfc.predict(X_test)
y_pred_dtc = dtc.predict(X_test)
y_pred_gbc = gbc.predict(X_test)

In [ ]:
rfc.feature_importances_

In [ ]:
df_main.columns[0:-1].ravel().values

In [ ]:
importance_df = pd.DataFrame({
    'features': df_main.columns[0:-1].values,
    'importance': rfc.feature_importances_
})

In [ ]:
importance_df.sort_values(by = 'importance', ascending = False, inplace = True)

In [ ]:
sns.barplot(data = importance_df, x = 'importance', y = 'features')
plt.title("Feature Importance", weight='bold')
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=300)
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

models = {
    "Logistic Regression": y_pred_lor,
    "Random Forest": y_pred_rfc,
    "Decision Tree": y_pred_dtc,
    "Gradient Boosting": y_pred_gbc
}

# 📊 1. Confusion Matrices
plt.figure(figsize=(12,8))

for i, (name, y_pred) in enumerate(models.items(), 1):
    cm = confusion_matrix(y_test, y_pred)

    plt.subplot(2,2,i)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(name)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")

plt.tight_layout()
plt.savefig("all_confusion_matrices.png", dpi=300)
plt.show()


# 📈 2. Metrics Table (HIGH RISK FOCUS)

results = []

for name, y_pred in models.items():
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision (High Risk)": precision_score(y_test, y_pred),
        "Recall (High Risk)": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred)
    })

results_df = pd.DataFrame(results)
print(results_df)

In [ ]:
y_prob = lor.predict_proba(X_test)[:,1]

In [ ]:
from sklearn.metrics import precision_recall_curve
import matplotlib.pyplot as plt

precision, recall, thresholds = precision_recall_curve(y_test, y_prob)

plt.plot(recall, precision)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.show()

In [ ]:
lor.predict_proba(X_test)[:,1]

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

# Get probabilities
y_prob_lor = lor.predict_proba(X_test)[:,1]
y_prob_rfc = rfc.predict_proba(X_test)[:,1]
y_prob_dtc = dtc.predict_proba(X_test)[:,1]
y_prob_gbc = gbc.predict_proba(X_test)[:,1]

models_prob = {
    "Logistic Regression": y_prob_lor,
    "Random Forest": y_prob_rfc,
    "Decision Tree": y_prob_dtc,
    "Gradient Boosting": y_prob_gbc
}

plt.figure(figsize=(8,6))

for name, y_prob in models_prob.items():
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)

    plt.plot(fpr, tpr, label=f"{name} (AUC = {roc_auc:.2f})")

# diagonal line (random model)
plt.plot([0,1], [0,1], linestyle='--')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison", weight='bold')
plt.legend()
plt.tight_layout()
plt.savefig("roc_all_models.png", dpi=300)
plt.show()